# 🏆 Turkish SLAM-ASR: Şampiyon Model (`t5`) + Zemberek Dil Modeli
### ⚡ Sıfır Eğitim Adımı — Doğrudan Model Yükleme, Zemberek LM Uygulama ve Karşılaştırmalı Değerlendirme

Bu notebook, tezimizin **en yüksek başarım gösteren şampiyon modeli olan `t5-ng-ta-ctx-tail-aug-lora`** üzerine **Zemberek Morfolojik Dil Modeli (Language Model - LM)** desteğini entegre etmek için özel olarak tasarlanmıştır.

---

### 🔬 Mimarinin 4 Temel Gücü
1. **Mimari Güç (`ng-ta-ctx`):** NeMo 1D derinlemesine ayrılabilir kapılı evrişimler ile 4 başlı Öz-Dikkat (Multi-Head Self-Attention) kollarını harmanlayan çift kollu bağlam duyarlı hibrit projektör.
2. **Dil Modeli Adaptasyon Gücü (`lora`):** 8 milyar parametreli üretken LLM omurgasına uygulanan PEFT LoRA ($r=16, \alpha=32$) ince ayarı.
3. **Akustik Dayanıklılık Gücü (`tail-aug`):** Konuşma bitişlerindeki ek düşmelerini önleyen kuyruk veri artırma (%3.17 CER rekoru).
4. **Morfolojik Dil Modeli Gücü (`lm` - Zemberek):** Türkçenin büyük/küçük ünlü uyumları ve ünsüz değişimlerini denetleyen kural tabanlı çözümleyici ve n-gram dil modeli ile ASR hipotezlerinin yeniden puanlanması (rescoring) ve düzeltilmesi (Görmez, 2024 LM desteği).

> ⚠️ **NOT:** Bu notebook'ta **hiçbir eğitim adımı (training) bulunmamaktadır.** Google Drive'da kayıtlı olan `t5` model ağırlıkları (`best_checkpoint`) doğrudan belleğe yüklenir veya daha önce üretilmiş tahmin dosyaları saniyeler içinde Zemberek LM ile yeniden puanlanır.


## 1. GPU ve Donanım Tespiti, Google Drive Bağlantısı
Colab oturumunuzun GPU hızlandırıcısını kontrol edin ve eğitilmiş model ağırlıklarına erişmek için Google Drive'ınızı bağlayın.


In [1]:
# 1. Adım: GPU Kontrolü ve Google Drive Bağlantısı
!nvidia-smi

import os
import sys
import torch

print(f"[*] PyTorch Sürümü: {torch.__version__}")
print(f"[*] CUDA Kullanılabilirliği: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"[*] Aktif GPU: {torch.cuda.get_device_name(0)}")
    print(f"[*] Toplam VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

# Google Drive'ı bağlayın (Colab ortamında çalışıyorsanız)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("[*] Google Drive başarıyla bağlandı.")
except Exception as e:
    print(f"[*] Google Colab ortamı dışındasınız veya Drive zaten bağlı: {e}")


/bin/bash: line 1: nvidia-smi: command not found
[*] PyTorch Sürümü: 2.14.1+cpu
[*] CUDA Kullanılabilirliği: False
Mounted at /content/drive
[*] Google Drive başarıyla bağlandı.


## 2. Çalışma Dizinini Hazırlama ve Bağımlılıkları Yükleme
Gerekli Python paketlerini (`transformers`, `peft`, `bitsandbytes`, `soundfile`, `librosa`, `jiwer`, `pyyaml`) yükleyin.


In [ ]:
# 2. Adım: Çalışma Dizinini Hazırlama ve Bağımlılıkları Yükleme
import os
import sys

# 1. slam_asr paketini ve champion-with-zemberek dizinini otomatik bağla
if not os.path.exists("slam_asr"):
    print("[*] slam_asr dizini mevcut konumda aranıyor...")
    
    # Eğer Colab ortamındaysak ve repo henüz klonlanmamışsa çek
    if not os.path.exists("/content/slam-asr-reimplement"):
        print("[*] GitHub deposu klonlanıyor (https://github.com/gokhanersoy/slam-asr-reimplement.git)...")
        !git clone https://github.com/gokhanersoy/slam-asr-reimplement.git /content/slam-asr-reimplement
    
    # Olası slam_asr konumlarını kontrol et
    candidates = [
        "/content/slam-asr-reimplement/champion-with-zemberek",
        "/content/slam-asr-reimplement/tr/tsc/t5-ng-ta-ctx-tail-aug-lora",
        "/content/slam-asr-reimplement",
        "./champion-with-zemberek",
        "../champion-with-zemberek",
    ]
    
    target_dir = None
    for c in candidates:
        if os.path.exists(os.path.join(c, "slam_asr")):
            target_dir = c
            break
            
    if target_dir:
        %cd {target_dir}
        if target_dir not in sys.path:
            sys.path.insert(0, target_dir)
        print(f"[✓] Çalışma dizini ayarlandı: {os.getcwd()}")
    else:
        # Eğer zip dosyası varsa aç
        if os.path.exists("champion-with-zemberek.zip"):
            !unzip -q champion-with-zemberek.zip
        elif os.path.exists("/content/drive/MyDrive/checkpoints/champion-with-zemberek.zip"):
            !cp /content/drive/MyDrive/checkpoints/champion-with-zemberek.zip . && unzip -q champion-with-zemberek.zip

# sys.path kontrolü
cur_dir = os.path.abspath(".")
if cur_dir not in sys.path:
    sys.path.insert(0, cur_dir)

print(f"[*] Aktif Python Çalışma Dizini: {cur_dir}")
print(f"[✓] slam_asr modül durumu: {'Mevcut ve Hazır!' if os.path.exists('slam_asr') else 'UYARI: slam_asr bulunamadı'}")

# 2. Gerekli Python Bağımlılıklarını Yükle
!pip install -q --upgrade pip
!pip install -q transformers>=4.40.0 accelerate>=0.28.0 peft>=0.7.0 bitsandbytes>=0.41.0 soundfile librosa jiwer pyyaml tqdm requests


## 3. Google Drive'daki Şampiyon Model (`t5`) Checkpoint'ini Otomatik Bulma
Google Drive'ınızdaki veya yerel depodaki `t5-ng-ta-ctx-tail-aug-lora` ağırlıklarını (`projector.pt` ve `lora_adapter`) otomatik tespit edin.


In [ ]:
# 3. Adım: Şampiyon Model (t5) Ağırlıklarını Otomatik Tespit Etme
import glob
import os

candidate_dirs = [
    # Google Colab Drive yolları (zaman damgalı klasörler ve doğrudan yollar)
    "/content/drive/MyDrive/checkpoints/*t5*",
    "/content/drive/MyDrive/checkpoints/*slam_asr*t5*",
    "/content/drive/MyDrive/checkpoints/t5-ng-ta-ctx-tail-aug-lora",
    "/content/drive/MyDrive/checkpoints/t5_ng_ta_ctx_tail_aug_lora",
    "/content/drive/MyDrive/_Akademik_/Doktora (Gazi)/checkpoints/*t5*",
    "/content/drive/MyDrive/_Akademik_/Doktora (Gazi)/checkpoints/*slam_asr*t5*",
    "/content/drive/MyDrive/outputs/slam_asr_turkish_colab/best_checkpoint",
    "/content/drive/MyDrive/outputs/slam_asr_turkish_colab",
    # Windows ve macOS Drive senkron yolları
    os.path.expanduser("~/Library/CloudStorage/GoogleDrive-*/My Drive/_Akademik_/Doktora (Gazi)/checkpoints/*t5*"),
    os.path.expanduser("~/Library/CloudStorage/GoogleDrive-*/Drive'ım/_Akademik_/Doktora (Gazi)/checkpoints/*t5*"),
    r"G:\Drive'ım\_Akademik_\Doktora (Gazi)\checkpoints\*t5*",
    "G:/Drive'ım/_Akademik_/Doktora (Gazi)/checkpoints/*t5*",
    # Yerel çalışma dizini yolları
    "./outputs/slam_asr_turkish_colab/best_checkpoint",
    "./outputs/best_checkpoint",
    "./checkpoints/t5-ng-ta-ctx-tail-aug-lora",
    "../tsc/t5-ng-ta-ctx-tail-aug-lora/outputs/best_checkpoint",
]

found_checkpoint_dir = None
for c in candidate_dirs:
    matches = sorted(glob.glob(c), reverse=True) if ("*" in c or "~" in c) else ([c] if os.path.exists(c) else [])
    for m in matches:
        if os.path.isdir(m):
            if os.path.exists(os.path.join(m, "projector.pt")):
                found_checkpoint_dir = m
                break
            best_sub = os.path.join(m, "best_checkpoint")
            if os.path.exists(os.path.join(best_sub, "projector.pt")):
                found_checkpoint_dir = best_sub
                break
    if found_checkpoint_dir:
        break

if found_checkpoint_dir:
    print(f"[✓] Şampiyon Model Checkpoint'i Başarıyla Bulundu: {found_checkpoint_dir}")
    print(f"    • Projector Ağırlığı: {os.path.join(found_checkpoint_dir, 'projector.pt')}")
    lora_sub = os.path.join(found_checkpoint_dir, "lora_adapter")
    if os.path.exists(lora_sub):
        print(f"    • LoRA Adaptörü:     {lora_sub}")
    else:
        print(f"    • LoRA Adaptörü:     {found_checkpoint_dir} (ana klasörde)")
else:
    print("[!] Uyarı: Standart konumlarda checkpoint otomatik bulunamadı.")
    print("    Lütfen checkpoint yolunu aşağıya manuel olarak girin:")
    found_checkpoint_dir = input("Checkpoint Klasör Yolu: ").strip() or "./outputs/best_checkpoint"


## 4. Zemberek Morfolojik Dil Modeli'ni (LM) Başlatma
Zemberek Dil Modeli, Türkçenin fonolojik ve morfolojik kurallarını (büyük ve küçük ünlü uyumu, ünsüz yumuşaması/benzeşmesi ve geçerli çekim eki zincirleri) denetleyen morfolojik analizci ve yumuşatılmış n-gram dil modelinden oluşur.

İster veri seti manifestosundan 3 saniyede sıfırdan n-gram tablosu oluşturabilir, isterseniz hazır kaydedilmiş model dosyasını yükleyebilirsiniz.


In [ ]:
# 4. Adım: Zemberek Morfolojik Dil Modeli'ni Başlatma
import os
import sys

# slam_asr modül yolunu kesin olarak güvenceye al
if not os.path.exists("slam_asr"):
    candidates = [
        "/content/slam-asr-reimplement/champion-with-zemberek",
        "/content/slam-asr-reimplement/tr/tsc/t5-ng-ta-ctx-tail-aug-lora",
        "/content/slam-asr-reimplement",
        os.path.abspath("."),
    ]
    for c in candidates:
        if os.path.exists(os.path.join(c, "slam_asr")):
            %cd {c}
            if c not in sys.path:
                sys.path.insert(0, c)
            break

cur_dir = os.path.abspath(".")
if cur_dir not in sys.path:
    sys.path.insert(0, cur_dir)

from slam_asr.utils.zemberek_lm import (
    ZemberekLanguageModel,
    TurkishMorphologyAnalyzer,
    check_vowel_harmony,
    fast_levenshtein
)

# Olası manifesto yollarını kontrol edin
manifest_candidates = [
    "data/issai_tsc/train.jsonl",
    "data/sample_train_tr.jsonl",
    "/content/drive/MyDrive/data/issai_tsc/train.jsonl",
    "/content/slam-asr-reimplement/champion-with-zemberek/data/sample_train_tr.jsonl",
]
train_manifest = None
for m in manifest_candidates:
    if os.path.exists(m):
        train_manifest = m
        break

print(f"[*] Zemberek Dil Modeli ilklendiriliyor...")
zemberek_lm = ZemberekLanguageModel(manifest_path=train_manifest, ngram_order=3)
print(f"[✓] Zemberek LM Başarıyla Hazırlandı!")
print(f"    • Sözlük Boyutu (Vocabulary): {len(zemberek_lm.vocab):,} sözcük")
print(f"    • Unigram Sayısı:              {len(zemberek_lm.unigrams):,}")
print(f"    • Bigram Sayısı:               {len(zemberek_lm.bigrams):,}")
print(f"    • Trigram Sayısı:              {len(zemberek_lm.trigrams):,}")

# Hızlı Morfolojik ve Fonolojik Doğrulama Testi
test_words = ["kitaplar", "evlerimizden", "çöplüklerimizdekilerinkindenmiş", "dinleyicileri"]
print("\n[*] Morfolojik ve Fonolojik Doğrulama Testi:")
for tw in test_words:
    harmony = check_vowel_harmony(tw)
    analyses = zemberek_lm.morphology.analyze(tw)
    root = analyses[0]['root'] if analyses else 'kök-bulunamadı'
    print(f"  • Kelime: {tw:<35} | Ünlü Uyumu: {str(harmony):<5} | Tespit Edilen Kök: {root}")


## 5. ⚡ HIZLI YOL: Mevcut `eval_results.jsonl` Tahminlerine Zemberek LM Uygulama (30 Saniye)
Eğer daha önce `t5` modelinizi test seti (ISSAI_TSC veya Mozilla Common Voice) üzerinde çalıştırdıysanız, modelin saatlerce/dakikalarca yeniden çıkarım (inference) yapmasını beklemenize gerek yoktur!

Bu adım, mevcut `eval_results.jsonl` dosyasını okur, her tahmin için Zemberek LM morfolojik düzeltmesini ve yeniden puanlamasını uygular; **saniyeler içinde** WER ve CER değişimini karşılaştırmalı olarak ekrana basar!


In [ ]:
# 5. Adım: Mevcut Tahmin Dosyasına Zemberek LM Uygulama (Saniyeler Sürer)
import glob
import os

eval_candidates = [
    # Colab Drive ve yerel tahmin dosyaları
    "eval_results_t5.jsonl",
    "eval_results.jsonl",
    (found_checkpoint_dir + "/eval_results.jsonl") if found_checkpoint_dir else "",
    "/content/drive/MyDrive/checkpoints/t5-ng-ta-ctx-tail-aug-lora/eval_results.jsonl",
    "/content/drive/MyDrive/checkpoints/t5-ng-ta-ctx-tail-aug-lora/eval_results_cv.jsonl",
]

target_eval_file = None
for ef in eval_candidates:
    if ef and os.path.exists(ef):
        target_eval_file = ef
        break

if target_eval_file:
    print(f"[*] Mevcut Tahmin Dosyası Bulundu: {target_eval_file}")
    cmd = f"python scripts/apply_zemberek_lm.py --input '{target_eval_file}' --output eval_results_champion_zemberek_lm.jsonl"
    if train_manifest:
        cmd += f" --train_manifest '{train_manifest}'"
    !{cmd}
else:
    print("[*] Bilgi: Önceden kaydedilmiş bir 'eval_results.jsonl' dosyası bulunamadı.")
    print("    Bir sonraki hücreye (6. Adım) geçerek modeli ses verileri üzerinde doğrudan çalıştırabilirsiniz.")


## 6. 🎙️ Uçtan Uca Model Değerlendirmesi: Test Seti Üzerinde `t5` + Zemberek LM Çalıştırma
Model ağırlıklarını belleğe yükleyip test seti (ISSAI_TSC_218 test seti veya Mozilla Common Voice test seti) üzerinde uçtan uca konuşma tanıma yapın. Zemberek LM otomatik olarak hem ham ASR tahminlerini hem de LM düzeltilmiş tahminleri hesaplayıp kıyaslar.


In [ ]:
# 6. Adım: Test Seti Üzerinde Uçtan Uca Değerlendirme
import os

test_manifest_candidates = [
    "data/issai_tsc/test.jsonl",
    "data/cv/test.jsonl",
    "data/sample_val_tr.jsonl",
    "/content/drive/MyDrive/data/issai_tsc/test.jsonl",
    "/content/drive/MyDrive/data/cv/test.jsonl",
]

test_manifest_path = None
for tm in test_manifest_candidates:
    if os.path.exists(tm):
        test_manifest_path = tm
        break

if not test_manifest_path:
    print("[!] Test manifestosu bulunamadı. Lütfen test dosyası yolunu girin:")
    test_manifest_path = input("Test Manifest Yolu: ").strip() or "data/sample_val_tr.jsonl"

print(f"[*] Seçilen Test Manifestosu: {test_manifest_path}")

# evaluate.py ile uçtan uca değerlendirmeyi başlatın
!python evaluate.py     --config config/colab_t4_turkish.yaml     --checkpoint "{found_checkpoint_dir}"     --manifest "{test_manifest_path}"     --output_file eval_results_champion_zemberek.jsonl     --use_zemberek_lm     --beam_size 4


## 7. 🔊 İnteraktif Çıkarım: Tekil Ses Dinleme ve Ham ASR vs Zemberek LM Karşılaştırması
Bir ses kaydını dinleyin; Şampiyon Modelimizin (`t5`) ham çıktısı ile Zemberek LM ile düzeltilmiş çıktısını yan yana inceleyin.


In [ ]:
# 7. Adım: Tekil Ses Dosyası ile İnteraktif Çıkarım
import IPython.display as ipd
import json
import glob
import os

sample_audio = None
# Önce test manifestosundan bir örnek arayalım
if test_manifest_path and os.path.exists(test_manifest_path):
    with open(test_manifest_path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                item = json.loads(line)
                wav = item.get("audio") or item.get("audio_filepath")
                if wav and os.path.exists(wav):
                    sample_audio = wav
                    break

# Eğer bulunamazsa toy audio dosyalarından seçelim
if not sample_audio or not os.path.exists(sample_audio):
    toy_candidates = glob.glob("data/toy_audio_tr/*.wav")
    if toy_candidates:
        sample_audio = toy_candidates[0]

if sample_audio and os.path.exists(sample_audio):
    print(f"[*] İncelenen Ses Dosyası: {sample_audio}")
    ipd.display(ipd.Audio(sample_audio))

    !python inference.py         --audio "{sample_audio}"         --config config/colab_t4_turkish.yaml         --checkpoint "{found_checkpoint_dir}"         --use_zemberek_lm
else:
    print("[!] Dinlenecek ses dosyası bulunamadı. Lütfen --audio parametresiyle kendi ses dosyanızı test edin.")


## 8. 🎤 Canlı Mikrofon Kaydı: Tarayıcıdan Konuş, Şampiyon Model + Zemberek LM ile Metne Çevir!
Google Colab üzerinde tarayıcınızın mikrofonunu kullanarak kendi sesinizi kaydedin; model anında transkribe etsin ve Zemberek LM ile dilbilgisel denetimini gerçekleştirsin.


In [ ]:
# 8. Adım: Colab Tarayıcı Mikrofonu ile Canlı Ses Kaydı ve Tanıma
from IPython.display import HTML, Javascript, display
import base64

RECORD_JS = """
const sleep = time => new Promise(resolve => setTimeout(resolve, time));
const b2text = blob => new Promise(resolve => {
  const reader = new FileReader();
  reader.onloadend = e => resolve(e.target.result);
  reader.readAsDataURL(blob);
});
var record = time => new Promise(async resolve => {
  stream = await navigator.mediaDevices.getUserMedia({ audio: true });
  recorder = new MediaRecorder(stream);
  chunks = [];
  recorder.ondataavailable = e => chunks.push(e.data);
  recorder.start();
  await sleep(time);
  recorder.onstop = async ()=>{
    blob = new Blob(chunks);
    text = await b2text(blob);
    resolve(text);
  };
  recorder.stop();
});
"""

def record_audio(sec=5, filename="live_recording.wav"):
    print(f"[*] {sec} saniye boyunca mikrofona Türkçe konuşun...")
    display(Javascript(RECORD_JS))
    from google.colab import output
    s = output.eval_js(f'record({sec * 1000})')
    b = base64.b64decode(s.split(',')[1])
    with open("temp.webm", "wb") as f:
        f.write(b)
    !ffmpeg -y -i temp.webm -ar 16000 -ac 1 -c:a pcm_s16le {filename} > /dev/null 2>&1
    print(f"[*] Ses başarıyla kaydedildi: {filename}")
    return filename

try:
    import IPython.display as ipd
    live_wav = record_audio(sec=5)
    display(ipd.Audio(live_wav))

    !python inference.py         --audio "{live_wav}"         --config config/colab_t4_turkish.yaml         --checkpoint "{found_checkpoint_dir}"         --use_zemberek_lm
except Exception as e:
    print(f"[*] Canlı mikrofon kaydı tamamlanamadı veya Colab dışındasınız: {e}")


## 9. 💾 Sonuçları Google Drive'a Otomatik Yedekleme
Elde edilen değerlendirme sonuçlarını (`eval_results_champion_zemberek.jsonl`), hata analizi raporunu ve logları Google Drive'ınıza güvenle yedekleyin.


In [ ]:
# 9. Adım: Sonuçları Google Drive'a Kaydetme
import shutil
import time
import os

timestamp = time.strftime("%Y%m%d_%H%M%S")
backup_dir = f"/content/drive/MyDrive/checkpoints/champion-with-zemberek-results_{timestamp}"

files_to_save = [
    "eval_results_champion_zemberek.jsonl",
    "eval_results_champion_zemberek_lm.jsonl",
]

saved_any = False
for f in files_to_save:
    if os.path.exists(f):
        os.makedirs(backup_dir, exist_ok=True)
        shutil.copy(f, os.path.join(backup_dir, f))
        print(f"[*] Kaydedildi: {os.path.join(backup_dir, f)}")
        saved_any = True

if saved_any:
    print(f"\n[✓] Tebrikler! Tüm Zemberek LM değerlendirme sonuçları başarıyla Drive'a yedeklendi: {backup_dir}")
else:
    print("[*] Henüz kaydedilecek bir sonuç dosyası üretilmedi.")
